In [ ]:
!pip -q install pandas numpy scikit-learn


استيراد المكتبات

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD

تحميل البيانات

In [ ]:
VISITS_PATH   = "/content/visits.csv"
PATIENTS_PATH = "/content/patients.csv"

visits   = pd.read_csv(VISITS_PATH)
patients = pd.read_csv(PATIENTS_PATH)

دمج البيانات

In [ ]:
df = visits.merge(patients, on="patient_id", how="left", suffixes=("", "_pat"))

target = "diagnosis_group"
df = df.dropna(subset=[target]).copy()

print("Shape:", df.shape)
print("Classes:", df[target].nunique())
print(df[target].value_counts())

Shape: (35000, 35)
Classes: 10
diagnosis_group
respiratory      6692
gi               4995
msk              3767
eye_ent          3596
neuro            3372
skin             3037
uti              2935
cardio           2762
general_fever    2437
other            1407
Name: count, dtype: int64


تحديد الخصائص (Features

In [ ]:
symptom_cols = [c for c in df.columns if c.startswith("symptom_")]
vital_cols   = [c for c in ["temp_c", "spo2", "heart_rate", "triage_level"] if c in df.columns]
cat_cols     = [c for c in ["visit_mode", "region_id", "age_group", "gender"] if c in df.columns]
bin_cols     = [c for c in ["chronic_diabetes", "chronic_hypertension", "risk_group"] if c in df.columns]

feature_cols = symptom_cols + vital_cols + cat_cols + bin_cols

X = df[feature_cols]
y = df[target]

تقسيم البيانات (Train/Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

 تجهيز المعالجة (Preprocessing)

In [ ]:
numeric_cols = symptom_cols + vital_cols + bin_cols
categorical_cols = cat_cols

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler(with_mean=False))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
])

In [ ]:
# نحسب عدد الـ features بعد preprocessing
X_tmp = preprocess.fit_transform(X_train)
n_features = X_tmp.shape[1]

# اختيار آمن تلقائي
n_comp = min(30, n_features - 1)

print("Total features after preprocessing:", n_features)
print("Using SVD components:", n_comp)

Total features after preprocessing: 52
Using SVD components: 30


Model

In [ ]:
model = Pipeline([
    ("pre", preprocess),
    ("svd", TruncatedSVD(n_components=n_comp)),
    ("clf", LogisticRegression(
        max_iter=3000,
        solver="saga",
        C=0.5,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

Train

In [ ]:
t0 = time.time()
model.fit(X_train, y_train)
train_time = time.time() - t0

print("Training time:", train_time)

Training time: 79.58091330528259


Predict

In [ ]:
t0 = time.time()
pred = model.predict(X_test)
pred_time = time.time() - t0

print("Prediction time:", pred_time)

Prediction time: 0.028440237045288086


Evaluation

In [ ]:
acc = accuracy_score(y_test, pred)
mf1 = f1_score(y_test, pred, average="macro")

print("Accuracy:", acc)
print("Macro F1:", mf1)

Accuracy: 0.8047142857142857
Macro F1: 0.7555297907049339


Detailed Results

In [ ]:
print("\nClassification Report:\n", classification_report(y_test, pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))


Classification Report:
                precision    recall  f1-score   support

       cardio       0.92      0.84      0.88       553
      eye_ent       0.76      0.78      0.77       719
general_fever       0.54      0.64      0.59       487
           gi       0.83      0.85      0.84       999
          msk       0.86      0.81      0.84       753
        neuro       0.79      0.78      0.78       675
        other       0.26      0.27      0.27       282
  respiratory       0.96      0.93      0.94      1338
         skin       0.82      0.78      0.80       607
          uti       0.82      0.88      0.85       587

     accuracy                           0.80      7000
    macro avg       0.76      0.76      0.76      7000
 weighted avg       0.81      0.80      0.81      7000

Confusion Matrix:
 [[ 462    9    7   14    8   15   19    3    6   10]
 [   0  563   30    9   14   23   23   28   11   18]
 [   3   25  311   48    6   17   19   16   22   20]
 [   5   11   55  850   

Check Predictions

In [ ]:
print("Unique predictions:", np.unique(pred))
print(pd.Series(pred).value_counts())

Unique predictions: ['cardio' 'eye_ent' 'general_fever' 'gi' 'msk' 'neuro' 'other'
 'respiratory' 'skin' 'uti']
respiratory      1304
gi               1018
eye_ent           737
msk               707
neuro             668
uti               626
skin              583
general_fever     573
cardio            500
other             284
Name: count, dtype: int64


اختبار عينة من البيانات نفسها

In [ ]:
# أخذ عينة من test set
sample = X_test.iloc[[0]]

# التنبؤ
prediction = model.predict(sample)

print("Sample input:")
display(sample)

print("Predicted class:", prediction[0])
print("Actual class:", y_test.iloc[0])

Sample input:


,symptom_fever,symptom_cough,symptom_sore_throat,symptom_diarrhea,symptom_vomiting,symptom_rash,symptom_headache,symptom_dizziness,symptom_chest_pain,symptom_palpitations,...,spo2,heart_rate,triage_level,visit_mode,region_id,age_group,gender,chronic_diabetes,chronic_hypertension,risk_group
29366,0,0,0,0,1,0,0,0,0,0,...,97,71,1,in_person,R008,20-39,M,0,0,0


Predicted class: eye_ent
Actual class: eye_ent


In [ ]:
new_sample = pd.DataFrame(columns=X.columns)

# تعبئة كل القيم بصفر (افتراضي)
new_sample.loc[0] = 0

اختبار عينة جديدة (Manual Input)

In [ ]:
# إنشاء DataFrame بنفس الأعمدة تماماً
new_sample = pd.DataFrame(columns=X.columns)

# تعبئة كل القيم بصفر (افتراضي)
new_sample.loc[0] = 0

In [ ]:
# القيم الحيوية
new_sample["temp_c"] = 38.5
new_sample["spo2"] = 95
new_sample["heart_rate"] = 100
new_sample["triage_level"] = 2

# أعراض (عدّل حسب الموجود عندك)
if "symptom_fever" in new_sample.columns:
    new_sample["symptom_fever"] = 1
if "symptom_cough" in new_sample.columns:
    new_sample["symptom_cough"] = 1

# بيانات فئوية (اختار من القيم الموجودة في الداتا)
new_sample["visit_mode"] = "walkin"
new_sample["region_id"] = 1
new_sample["age_group"] = "adult"
new_sample["gender"] = "male"

# أمراض مزمنة
new_sample["chronic_diabetes"] = 0
new_sample["chronic_hypertension"] = 1
new_sample["risk_group"] = 1

In [ ]:
prediction = model.predict(new_sample)

print("Predicted diagnosis:", prediction[0])

Predicted diagnosis: respiratory


In [ ]:
print(f"Predicted: {prediction} | Actual: {true_label} | Correct: {prediction == true_label}")

Predicted: ['respiratory'] | Actual: eye_ent | Correct: [False]


اختبار أكثر من عينة

In [ ]:
for i in range(5):  # أول 5 عينات
    sample = X_test.iloc[[i]]
    true_label = y_test.iloc[i]
    pred = model.predict(sample)[0]

    print(f"[{i}] Pred: {pred} | Actual: {true_label} | {'✅' if pred==true_label else '❌'}")

[0] Pred: eye_ent | Actual: eye_ent | ✅
[1] Pred: gi | Actual: gi | ✅
[2] Pred: respiratory | Actual: respiratory | ✅
[3] Pred: uti | Actual: uti | ✅
[4] Pred: gi | Actual: general_fever | ❌
